# Membuat korpus QA Bencana BNPB

Notebook ini membuat korpus untuk **QA tanpa LLM dan QA RAG** dari sembilan topik buku BNPB.
Aturan halaman dan tata letak ditulis langsung di notebook.

1. Letakkan PDF asli di folder `knowledge_source/` pada folder kerja notebook.
2. Pastikan library pada `requirements.txt` sudah terpasang dan pilih kernel Python yang sesuai.
3. Jalankan **Run all**. Hasil disimpan di `output/`: `classic_passages.jsonl`, `rag_chunks.jsonl`, dan `corpus.md`. Tidak ada file manifest JSON.

Di lokal, jalankan notebook dari folder `corpus/`. Di Colab, buat folder `knowledge_source/` melalui panel **Files**, lalu unggah PDF ke folder tersebut. Jika memakai Google Drive, mount Drive terlebih dahulu dan ubah `PROJECT_DIR` ke folder yang berisi `knowledge_source/`.

CPU sudah cukup. Internet diperlukan untuk pemasangan library dan pengambilan tokenizer pada penggunaan pertama. PDF tidak diunduh otomatis.

Pada hasil JSONL, `text` berisi teks buku dan `retrieval_text` menambahkan konteks bagian untuk pencarian. `source` berisi nama file dan halaman. Chunk RAG juga memiliki `unit_ids` dan `token_count`. Status isi tetap `pending_group_review` sampai diperiksa kelompok.

## 1. Pengaturan dan PDF sumber

Biarkan `PDF_FILENAME = None` jika folder hanya berisi satu PDF. Jika ada beberapa PDF, isi nama file yang dipilih. Hash SHA256 memastikan PDF sesuai dengan aturan koordinat di notebook ini.

In [1]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import re
import unicodedata

import pymupdf
from transformers import AutoTokenizer

PROJECT_DIR = Path.cwd()  # Ubah jika folder proyek berada di lokasi lain.
KNOWLEDGE_DIR = PROJECT_DIR / "knowledge_source"
OUTPUT_DIR = PROJECT_DIR / "output"
PDF_FILENAME = None
MAX_TOKENS = 350
TOKENIZER_ID = "intfloat/multilingual-e5-small"
TOKENIZER_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
EXPECTED_PDF_SHA256 = "2d96ec6767f06f42a88669e48ab14974042cda5accbbcf3f81e08645dca84f4a"
EXCLUDE_IDS = set()  # Opsional: isi ID unit yang ingin dikeluarkan, lalu Run all.

if not KNOWLEDGE_DIR.is_dir():
    raise FileNotFoundError(f"Buat folder {KNOWLEDGE_DIR} dan letakkan PDF BNPB asli di dalamnya.")

pdf_files = []
for file_path in KNOWLEDGE_DIR.iterdir():
    if file_path.is_file() and file_path.suffix.lower() == ".pdf":
        pdf_files.append(file_path)
pdf_files.sort()

if PDF_FILENAME is None:
    if len(pdf_files) != 1:
        raise ValueError(f"Ditemukan {len(pdf_files)} PDF. Isi PDF_FILENAME dengan nama file yang dipilih.")
    PDF_PATH = pdf_files[0]
else:
    if Path(PDF_FILENAME).name != PDF_FILENAME:
        raise ValueError("PDF_FILENAME harus berupa nama file di dalam knowledge_source/.")
    PDF_PATH = KNOWLEDGE_DIR / PDF_FILENAME

pdf_bytes = PDF_PATH.read_bytes()
pdf_sha256 = hashlib.sha256(pdf_bytes).hexdigest()
if pdf_sha256 != EXPECTED_PDF_SHA256:
    raise ValueError("PDF berbeda dari salinan BNPB 98 halaman yang dipakai proyek. Gunakan PDF asli yang sama.")

output_path = OUTPUT_DIR.resolve()
source_path = KNOWLEDGE_DIR.resolve()
if output_path.is_relative_to(source_path) or source_path.is_relative_to(output_path):
    raise ValueError("OUTPUT_DIR harus terpisah dari knowledge_source/.")

print(f"PDF: {PDF_PATH.name}")
print("SHA256 cocok.")
print(f"Output: {OUTPUT_DIR}")

/home/fast9/Secondary/CS5/nlp/ta-nlp/venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PDF: Buku Saku Bencana BNPB.pdf
SHA256 cocok.
Output: /home/fast9/Secondary/CS5/nlp/ta-nlp/corpus/output


## 2. Pemetaan isi buku

Nomor halaman berikut adalah **nomor halaman PDF, dimulai dari satu**. Nomor halaman buku adalah nomor PDF dikurangi dua. Hanya rentang halaman berikut yang masuk korpus. Fase pascabencana tidak dibuat untuk karhutla atau COVID-19 karena tidak tersedia sebagai subbagian buku.

In [2]:
SECTIONS = [
    (10, 10, 'Ancaman geologi', 'UMUM', 'GEOLOGI'),
    (11, 12, 'Ancaman hidrometeorologi', 'UMUM', 'HIDROMETEOROLOGI'),
    (13, 14, 'Penanggulangan bencana dan posko', 'UMUM', 'LEMBAGA'),
    (16, 19, 'Gempa bumi', 'GEMPA', 'BENCANA_UTAMA'),
    (20, 26, 'Tsunami', 'TSUNAMI', 'BENCANA_UTAMA'),
    (27, 33, 'Erupsi gunungapi', 'ERUPSI', 'BENCANA_UTAMA'),
    (34, 41, 'Banjir', 'BANJIR', 'BENCANA_UTAMA'),
    (42, 46, 'Tanah longsor', 'LONGSOR', 'BENCANA_UTAMA'),
    (47, 50, 'Puting beliung', 'PUTING', 'BENCANA_UTAMA'),
    (51, 54, 'Kekeringan', 'KERING', 'BENCANA_UTAMA'),
    (55, 58, 'Kebakaran hutan dan lahan', 'KARHUTLA', 'BENCANA_UTAMA'),
    (59, 62, 'Pandemi COVID-19', 'COVID', 'BENCANA_UTAMA'),
    (64, 68, 'Kesiapsiagaan umum', 'UMUM', 'KESIAPSIAGAAN'),
    (69, 69, 'Mitigasi untuk penyandang disabilitas', 'UMUM', 'DISABILITAS'),
    (71, 75, 'Mitigasi vegetasi', 'UMUM', 'VEGETASI'),
    (76, 76, 'Kesiapsiagaan hewan peliharaan', 'UMUM', 'HEWAN'),
    (77, 80, 'Klasifikasi bahan berbahaya dan beracun', 'UMUM', 'B3'),
    (82, 82, 'Rambu dan papan informasi bencana', 'UMUM', 'RAMBU'),
    (83, 83, 'Aplikasi gawai dalam buku', 'UMUM', 'APLIKASI'),
    (84, 91, 'Kontak dan BPBD dalam buku', 'UMUM', 'KONTAK'),
    (92, 92, 'Daftar singkatan', 'UMUM', 'SINGKATAN'),
]

DEFAULT_PHASE = {18: 'SAAT', 19: 'SAAT', 22: 'PRA', 23: 'SAAT', 24: 'SAAT', 25: 'PASCA', 26: 'PASCA', 32: 'SAAT', 33: 'PASCA', 36: 'PRA', 37: 'PRA', 38: 'SAAT', 39: 'SAAT', 40: 'PASCA', 41: 'PASCA', 43: 'PRA', 44: 'PRA', 45: 'PRA', 46: 'SAAT', 48: 'PRA', 49: 'SAAT', 50: 'SAAT', 52: 'PRA', 53: 'SAAT', 54: 'PASCA', 57: 'PRA', 58: 'SAAT', 62: 'PRA'}
PHASE_CHANGES = {17: [[156, 'PRA']], 19: [[227, 'PASCA']], 31: [[206, 'PRA']], 35: [[195, 'PRA']], 46: [[144, 'PASCA']], 48: [[170, 'SAAT']], 50: [[205, 'PASCA']], 61: [[226, 'PRA']]}
SKIP_EXACT = {18: ['Di dalam bangunan, seperti rumah, sekolah ataupun', 'bangunan bertingkat:'], 19: ['Di dalam mobil:']}
HEADINGS = {'Erupsi Gunungapi', 'Hidrometeorologi', 'Kekeringan', 'Gempa Bumi', 'Banjir', 'MITIGASI VEGETASI', 'Gejala', 'Rencana', 'COVID-19', 'Puting Beliung', 'Kesiapsiagaan', 'Cara Penularan', 'Tsunami', 'di Indonesia', 'DAFTAR SINGKATAN', 'PANDEMI', 'Tanah Longsor', 'Kebakaran Hutan dan Lahan', 'Penanggulangan Bencana', 'Geologi'}
PLANT_CONTEXT = {71: 'Tanaman vetiver', 72: 'Pohon kelapa', 73: 'Pohon beringin', 74: 'Cemara laut', 75: 'Bakau'}
HAZARDS = {"GEMPA", "TSUNAMI", "ERUPSI", "BANJIR", "LONGSOR", "PUTING", "KERING", "KARHUTLA", "COVID"}

## 3. Aturan halaman dengan tata letak khusus

Setiap `rect` adalah koordinat area teks pada halaman PDF. `parts` menggabungkan beberapa area, sedangkan `label` menjaga hubungan antar kolom tabel. `text_override` memperbaiki transkripsi atau urutan baca, dengan alasan pada `note`.

Data ini diperlukan untuk tabel, infografik, daftar tas siaga, B3, rambu, aplikasi, dan kontak. Urutan teks yang disimpan dalam PDF tidak selalu sesuai dengan urutan baca.

In [3]:
LAYOUT = {
    12: [
        {'parts': [{'rect': [165, 65, 270, 112]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 145, 270, 183]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 183, 270, 210]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [25, 230, 175, 264], 'label': 'Pola dan wilayah'}, {'rect': [178, 230, 270, 264], 'label': 'Puncak musim hujan'}], 'phase': 'UMUM', 'subsection': 'Pola hujan'},
        {'parts': [{'rect': [25, 267, 175, 324], 'label': 'Pola dan wilayah'}, {'rect': [178, 267, 270, 324], 'label': 'Puncak musim hujan'}], 'phase': 'UMUM', 'subsection': 'Pola hujan'},
        {'parts': [{'rect': [25, 329, 175, 358], 'label': 'Pola dan wilayah'}, {'rect': [178, 329, 270, 358], 'label': 'Puncak musim hujan'}], 'phase': 'UMUM', 'subsection': 'Pola hujan'},
    ],
    14: [
        {'parts': [{'rect': [20, 14, 270, 60]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 60, 270, 106]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 109, 270, 135]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [10, 148, 275, 370]}], 'phase': 'UMUM', 'subsection': 'Struktur posko', 'text_override': 'Komandan; Wakil Komandan; Sekretariat; Bagian Perencanaan; Bagian Data, Informasi dan Humas; Perwakilan Mitra Kerja; Bidang Operasi.', 'note': 'Daftar label pada diagram; hubungan hierarkis tidak disimpulkan dari urutan teks. PDF tetap rujukan hubungan garis.'},
    ],
    17: [
        {'parts': [{'rect': [75, 45, 270, 112]}, {'rect': [22, 111, 270, 142]}], 'phase': 'UMUM', 'subsection': 'Informasi dan lembaga', 'text_override': 'Sampai saat ini, belum ada ahli dan institusi yang mampu memprediksi kapan terjadinya gempa bumi. Institusi yang berwenang untuk mengeluarkan informasi kejadian gempa bumi adalah BMKG. Anda dapat mengetahui informasi dari berbagai parameter mengenai besaran suatu gempa bumi, titik pusat gempa bumi, kedalaman,dan potensi tsunami dari laman (www.bmkg.go.id) atau pun aplikasi gawai BMKG berbasis android atau IOS.', 'note': 'Menggabungkan paragraf terpotong antarblok; ilustrasi URL monitor tidak diduplikasi. Penyatuan bmkg.go.id mengembalikan URL pada sumber.'},
        {'parts': [{'rect': [20, 185, 275, 223]}], 'phase': 'PRA'},
        {'parts': [{'rect': [20, 232, 275, 279]}], 'phase': 'PRA'},
        {'parts': [{'rect': [20, 285, 275, 314]}], 'phase': 'PRA'},
        {'parts': [{'rect': [20, 323, 275, 363]}], 'phase': 'PRA'},
        {'parts': [{'rect': [20, 368, 275, 405]}], 'phase': 'PRA'},
    ],
    25: [
        {'parts': [{'rect': [20, 88, 275, 113]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 128, 275, 154]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 169, 275, 195]}], 'phase': 'PASCA', 'text_override': 'Jauhi area yang tergenang dan rusak sampai ada informasi aman dari pihak berwenang.', 'note': 'Mengabaikan glyph r lepas yang tidak tampak sebagai bagian kalimat pada halaman visual.'},
        {'parts': [{'rect': [20, 210, 275, 236]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 250, 275, 277]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 291, 275, 318]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 332, 275, 359]}], 'phase': 'PASCA'},
    ],
    26: [
        {'parts': [{'rect': [20, 25, 275, 43]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 62, 275, 88]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 103, 275, 129]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 147, 275, 165]}], 'phase': 'PASCA', 'text_override': 'Tetap di luar gedung yang masih dikelilingi genangan air.', 'note': 'Menyatukan glyph awal T yang disimpan terpisah dari etap pada PDF; ejaan cocok dengan tampilan sumber.'},
        {'parts': [{'rect': [20, 184, 275, 210]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 221, 275, 256]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 270, 275, 287]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 306, 275, 333]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 347, 275, 374]}], 'phase': 'PASCA'},
    ],
    35: [
        {'parts': [{'rect': [20, 42, 275, 85]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 85, 275, 128]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 128, 275, 170]}], 'phase': 'UMUM'},
        {'parts': [{'rect': [20, 230, 275, 266]}], 'phase': 'PRA'},
        {'parts': [{'rect': [20, 280, 275, 308]}], 'phase': 'PRA', 'text_override': 'Mengetahui tingkat kerentanan tempat tinggal kita, apakah berada di zona rawan banjir.', 'note': 'Menyatukan glyph akhir r yang terpisah dari banji pada PDF.'},
        {'parts': [{'rect': [20, 324, 275, 343]}], 'phase': 'PRA'},
        {'parts': [{'rect': [20, 360, 275, 388]}], 'phase': 'PRA'},
    ],
    41: [
        {'parts': [{'rect': [20, 49, 275, 85]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 103, 275, 122]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 141, 275, 176]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 190, 275, 217]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 231, 275, 259]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 272, 275, 300]}], 'phase': 'PASCA'},
        {'parts': [{'rect': [20, 318, 275, 338]}], 'phase': 'PASCA', 'text_override': 'Terlibat dalam kaporitisasi sumur gali.', 'note': 'Menyatukan glyph awal T yang disimpan terpisah dari erlibat pada PDF.'},
        {'parts': [{'rect': [20, 355, 275, 383]}], 'phase': 'PASCA'},
    ],
    29: [
        {'parts': [{'rect': [20, 40, 270, 78]}], 'phase': 'UMUM', 'subsection': 'Status gunungapi dan kawasan rawan'},
        {'parts': [{'rect': [15, 181, 54, 228], 'label': 'Level'}, {'rect': [54, 181, 94, 228], 'label': 'Status'}, {'rect': [94, 181, 272, 228], 'label': 'Penjelasan'}], 'phase': 'UMUM', 'subsection': 'Status aktivitas gunungapi'},
        {'parts': [{'rect': [15, 240, 54, 295], 'label': 'Level'}, {'rect': [54, 240, 94, 295], 'label': 'Status'}, {'rect': [94, 240, 272, 295], 'label': 'Penjelasan'}], 'phase': 'UMUM', 'subsection': 'Status aktivitas gunungapi'},
        {'parts': [{'rect': [15, 305, 54, 342], 'label': 'Level'}, {'rect': [54, 305, 94, 342], 'label': 'Status'}, {'rect': [94, 305, 272, 342], 'label': 'Penjelasan'}], 'phase': 'UMUM', 'subsection': 'Status aktivitas gunungapi'},
        {'parts': [{'rect': [15, 355, 54, 392], 'label': 'Level'}, {'rect': [54, 355, 94, 392], 'label': 'Status'}, {'rect': [94, 355, 272, 392], 'label': 'Penjelasan'}], 'phase': 'UMUM', 'subsection': 'Status aktivitas gunungapi'},
    ],
    30: [
        {'parts': [{'rect': [20, 13, 270, 32]}], 'phase': 'UMUM', 'subsection': 'Kawasan rawan bencana'},
        {'parts': [{'rect': [90, 76, 273, 120]}], 'phase': 'UMUM', 'subsection': 'Kawasan rawan bencana'},
        {'parts': [{'rect': [90, 124, 273, 235]}], 'phase': 'UMUM', 'subsection': 'Kawasan rawan bencana'},
        {'parts': [{'rect': [90, 239, 273, 377]}], 'phase': 'UMUM', 'subsection': 'Kawasan rawan bencana'},
    ],
    56: [
        {'parts': [{'rect': [20, 12, 270, 51]}], 'phase': 'UMUM', 'subsection': 'Kualitas udara PM10'},
        {'parts': [{'rect': [150, 78, 270, 147]}], 'phase': 'UMUM', 'subsection': 'Sumber informasi dalam buku', 'quality_flags': ['TIME_SENSITIVE'], 'text_override': 'Informasi kualitas udara di beberapa tempat dapat diakses di laman berikut: http://www.bmkg.go.id/kualitas-udara/informasi-partikulat-pm10.bmkg', 'note': 'Penyatuan URL yang terbungkus antarbaris; tidak mengubah domain atau path.'},
        {'parts': [{'rect': [55, 85, 115, 115]}], 'phase': 'UMUM', 'subsection': 'Indikator kualitas udara pada buku', 'quality_flags': ['SOURCE_AMBIGUOUS', 'TIME_SENSITIVE']},
        {'parts': [{'rect': [55, 190, 115, 220]}], 'phase': 'UMUM', 'subsection': 'Indikator kualitas udara pada buku', 'quality_flags': ['SOURCE_AMBIGUOUS', 'TIME_SENSITIVE']},
        {'parts': [{'rect': [170, 185, 225, 224]}], 'phase': 'UMUM', 'subsection': 'Indikator kualitas udara pada buku', 'quality_flags': ['SOURCE_AMBIGUOUS', 'TIME_SENSITIVE']},
        {'parts': [{'rect': [50, 292, 116, 332]}], 'phase': 'UMUM', 'subsection': 'Indikator kualitas udara pada buku', 'quality_flags': ['SOURCE_AMBIGUOUS', 'TIME_SENSITIVE']},
        {'parts': [{'rect': [178, 292, 235, 332]}], 'phase': 'UMUM', 'subsection': 'Indikator kualitas udara pada buku', 'quality_flags': ['SOURCE_AMBIGUOUS', 'TIME_SENSITIVE']},
    ],
    61: [
        {'parts': [{'rect': [28, 82, 273, 160]}], 'phase': 'UMUM', 'subsection': 'Gejala dalam buku'},
        {'parts': [{'rect': [28, 160, 273, 203]}], 'phase': 'UMUM', 'subsection': 'Kasus tanpa gejala dalam buku'},
        {'parts': [{'rect': [20, 247, 270, 279]}], 'phase': 'PRA', 'subsection': 'Tiga langkah utama pencegahan'},
        {'parts': [{'rect': [100, 282, 265, 323]}], 'phase': 'PRA', 'subsection': 'Tiga langkah utama pencegahan'},
        {'parts': [{'rect': [100, 326, 185, 377]}], 'phase': 'PRA', 'subsection': 'Tiga langkah utama pencegahan'},
        {'parts': [{'rect': [100, 378, 210, 404]}], 'phase': 'PRA', 'subsection': 'Tiga langkah utama pencegahan'},
    ],
    64: [
        {'parts': [{'rect': [20, 109, 270, 165]}], 'phase': 'UMUM', 'subsection': 'Pengertian kesiapsiagaan'},
        {'parts': [{'rect': [20, 167, 270, 232]}], 'phase': 'UMUM', 'subsection': 'Hari Kesiapsiagaan Bencana', 'text_override': 'BNPB menetapkan 26 APRIL sebagai Hari Kesiapsiagaan Bencana dan mengharapkan partisipasi Anda dan semua pihak untuk melakukan latihan kesiapsiagaan.', 'note': 'Menyusun ulang empat label infografik sesuai urutan baca visual; tidak menambah fakta.'},
    ],
    65: [
        {'parts': [{'rect': [65, 42, 273, 79]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
        {'parts': [{'rect': [60, 92, 275, 117]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
        {'parts': [{'rect': [60, 148, 275, 176]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
        {'parts': [{'rect': [60, 193, 275, 231]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
        {'parts': [{'rect': [60, 244, 275, 281]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
        {'parts': [{'rect': [60, 303, 275, 322]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
        {'parts': [{'rect': [60, 353, 275, 373]}], 'phase': 'UMUM', 'subsection': 'Upaya kesiapsiagaan'},
    ],
    66: [
        {'parts': [{'rect': [45, 64, 270, 137]}], 'phase': 'UMUM', 'subsection': 'Rencana kesiapsiagaan'},
        {'parts': [{'rect': [70, 141, 270, 174]}], 'phase': 'UMUM', 'subsection': 'Rencana darurat keluarga'},
        {'parts': [{'rect': [20, 177, 270, 372]}], 'phase': 'UMUM', 'subsection': 'Rencana darurat keluarga'},
    ],
    67: [
        {'parts': [{'rect': [30, 89, 265, 151]}], 'phase': 'UMUM', 'subsection': 'Pengertian dan tujuan tas siaga bencana'},
        {'parts': [{'rect': [10, 194, 98, 234]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Surat-surat penting'},
        {'parts': [{'rect': [10, 233, 65, 293]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Pakaian untuk 3 hari', 'text_override': 'Pakaian Untuk 3 Hari seperti: pakaian dalam, celana panjang, jaket, selimut, handuk, jas hujan, dsb.', 'note': 'Memisahkan deskripsi barang di sisi kiri dari label duplikat dalam ilustrasi tas. Transkripsi mengikuti teks visual sumber.'},
        {'parts': [{'rect': [10, 295, 70, 314]}, {'rect': [10, 314, 85, 331]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Makanan ringan tahan lama', 'text_override': 'Makanan Ringan Tahan Lama seperti: mie instant, biskuit, abon, coklat, dsb.', 'note': 'Memisahkan deskripsi barang di sisi kiri dari label duplikat dalam ilustrasi tas. Transkripsi mengikuti teks visual sumber.'},
        {'parts': [{'rect': [10, 333, 99, 369]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Air minum'},
        {'parts': [{'rect': [10, 369, 99, 398]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Kotak obat-obatan atau P3K'},
        {'parts': [{'rect': [211, 194, 275, 239]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Alat bantu penerangan'},
        {'parts': [{'rect': [215, 240, 275, 292]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Uang'},
        {'parts': [{'rect': [212, 293, 275, 330]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Peluit'},
        {'parts': [{'rect': [208, 331, 275, 368]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Masker'},
        {'parts': [{'rect': [188, 369, 275, 399]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Perlengkapan mandi'},
        {'parts': [{'rect': [100, 361, 185, 400]}], 'phase': 'UMUM', 'subsection': 'Kebutuhan dasar tas siaga bencana untuk 3 hari', 'context': 'Radio atau ponsel'},
    ],
    69: [
        {'parts': [{'rect': [20, 75, 275, 100]}], 'phase': 'UMUM', 'subsection': 'Disabilitas dan risiko bencana'},
        {'parts': [{'rect': [49, 102, 145, 132]}], 'phase': 'UMUM', 'subsection': 'Disabilitas dan risiko bencana'},
        {'parts': [{'rect': [192, 102, 275, 132]}], 'phase': 'UMUM', 'subsection': 'Disabilitas dan risiko bencana'},
        {'parts': [{'rect': [49, 138, 140, 167]}], 'phase': 'UMUM', 'subsection': 'Disabilitas dan risiko bencana'},
        {'parts': [{'rect': [4, 274, 102, 391]}], 'phase': 'PRA', 'subsection': 'Mitigasi untuk penyandang disabilitas'},
        {'parts': [{'rect': [104, 286, 202, 399]}], 'phase': 'SAAT', 'subsection': 'Mitigasi untuk penyandang disabilitas'},
        {'parts': [{'rect': [203, 315, 278, 374]}], 'phase': 'PASCA', 'subsection': 'Mitigasi untuk penyandang disabilitas'},
    ],
    76: [
        {'parts': [{'rect': [15, 57, 278, 160]}], 'phase': 'UMUM', 'subsection': 'Perlindungan hewan peliharaan'},
        {'parts': [{'rect': [27, 185, 160, 218]}], 'phase': 'UMUM', 'subsection': 'Persiapan hewan peliharaan'},
        {'parts': [{'rect': [25, 219, 135, 256]}], 'phase': 'UMUM', 'subsection': 'Persiapan hewan peliharaan'},
        {'parts': [{'rect': [25, 266, 140, 306]}], 'phase': 'UMUM', 'subsection': 'Persiapan hewan peliharaan'},
        {'parts': [{'rect': [177, 170, 278, 261]}], 'phase': 'UMUM', 'subsection': 'Persiapan hewan peliharaan'},
        {'parts': [{'rect': [214, 265, 270, 306]}], 'phase': 'UMUM', 'subsection': 'Persiapan hewan peliharaan'},
        {'parts': [{'rect': [60, 323, 150, 338]}], 'phase': 'UMUM', 'subsection': 'Perlengkapan tas darurat hewan peliharaan'},
        {'parts': [{'rect': [60, 338, 150, 357]}], 'phase': 'UMUM', 'subsection': 'Perlengkapan tas darurat hewan peliharaan'},
        {'parts': [{'rect': [60, 357, 151, 383]}], 'phase': 'UMUM', 'subsection': 'Perlengkapan tas darurat hewan peliharaan'},
        {'parts': [{'rect': [187, 319, 275, 346]}], 'phase': 'UMUM', 'subsection': 'Perlengkapan tas darurat hewan peliharaan'},
        {'parts': [{'rect': [187, 346, 275, 363]}], 'phase': 'UMUM', 'subsection': 'Perlengkapan tas darurat hewan peliharaan'},
        {'parts': [{'rect': [187, 365, 275, 379]}], 'phase': 'UMUM', 'subsection': 'Perlengkapan tas darurat hewan peliharaan'},
    ],
    77: [
        {'parts': [{'rect': [40, 109, 273, 185]}], 'phase': 'UMUM', 'subsection': 'Definisi bahan berbahaya dan beracun'},
        {'parts': [{'rect': [150, 210, 270, 254]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
        {'parts': [{'rect': [160, 320, 270, 365]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
    ],
    78: [
        {'parts': [{'rect': [10, 45, 275, 155]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
        {'parts': [{'rect': [10, 165, 275, 270]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
        {'parts': [{'rect': [10, 280, 275, 389]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
    ],
    79: [
        {'parts': [{'rect': [10, 45, 275, 155]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
        {'parts': [{'rect': [10, 165, 275, 270]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
        {'parts': [{'rect': [10, 280, 275, 389]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
    ],
    80: [
        {'parts': [{'rect': [125, 70, 273, 103]}], 'phase': 'UMUM', 'subsection': 'Kelas bahan berbahaya dan beracun'},
        {'parts': [{'rect': [38, 148, 273, 252]}], 'phase': 'UMUM', 'subsection': 'Dasar hukum yang dikutip buku', 'quality_flags': ['HISTORICAL_RECORD', 'TIME_SENSITIVE']},
    ],
    84: [
        {'parts': [{'rect': [47, 45, 175, 85], 'label': 'Kontak'}, {'rect': [175, 45, 278, 85], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 86, 175, 103], 'label': 'Kontak'}, {'rect': [175, 86, 278, 103], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 104, 175, 130], 'label': 'Kontak'}, {'rect': [175, 104, 278, 130], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 131, 175, 147], 'label': 'Kontak'}, {'rect': [175, 131, 278, 147], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 148, 175, 164], 'label': 'Kontak'}, {'rect': [175, 148, 278, 164], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 165, 175, 181], 'label': 'Kontak'}, {'rect': [175, 165, 278, 181], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 182, 175, 197], 'label': 'Kontak'}, {'rect': [175, 182, 278, 197], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 198, 175, 224], 'label': 'Kontak'}, {'rect': [175, 198, 278, 224], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 225, 175, 241], 'label': 'Kontak'}, {'rect': [175, 225, 278, 241], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 242, 175, 258], 'label': 'Kontak'}, {'rect': [175, 242, 278, 258], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 259, 175, 274], 'label': 'Kontak'}, {'rect': [175, 259, 278, 274], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 275, 175, 291], 'label': 'Kontak'}, {'rect': [175, 275, 278, 291], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 292, 175, 308], 'label': 'Kontak'}, {'rect': [175, 292, 278, 308], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 309, 175, 334], 'label': 'Kontak'}, {'rect': [175, 309, 278, 334], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 335, 175, 360], 'label': 'Kontak'}, {'rect': [175, 335, 278, 360], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 361, 175, 380], 'label': 'Kontak'}, {'rect': [175, 361, 278, 380], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    85: [
        {'parts': [{'rect': [47, 95, 138, 184], 'label': 'Provinsi dan alamat'}, {'rect': [138, 95, 278, 184], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 185, 138, 238], 'label': 'Provinsi dan alamat'}, {'rect': [138, 185, 278, 238], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 239, 138, 292], 'label': 'Provinsi dan alamat'}, {'rect': [138, 239, 278, 292], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 294, 138, 339], 'label': 'Provinsi dan alamat'}, {'rect': [138, 294, 278, 339], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 340, 138, 390], 'label': 'Provinsi dan alamat'}, {'rect': [138, 340, 278, 390], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    86: [
        {'parts': [{'rect': [47, 33, 138, 75], 'label': 'Provinsi dan alamat'}, {'rect': [138, 33, 278, 75], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 75, 138, 127], 'label': 'Provinsi dan alamat'}, {'rect': [138, 75, 278, 127], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 127, 138, 161], 'label': 'Provinsi dan alamat'}, {'rect': [138, 127, 278, 161], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 161, 138, 211], 'label': 'Provinsi dan alamat'}, {'rect': [138, 161, 278, 211], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 211, 138, 294], 'label': 'Provinsi dan alamat'}, {'rect': [138, 211, 278, 294], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 294, 138, 367], 'label': 'Provinsi dan alamat'}, {'rect': [138, 294, 278, 367], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    87: [
        {'parts': [{'rect': [47, 79, 138, 167], 'label': 'Provinsi dan alamat'}, {'rect': [138, 79, 278, 167], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 167, 138, 232], 'label': 'Provinsi dan alamat'}, {'rect': [138, 167, 278, 232], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 232, 138, 329], 'label': 'Provinsi dan alamat'}, {'rect': [138, 232, 278, 329], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 329, 138, 394], 'label': 'Provinsi dan alamat'}, {'rect': [138, 329, 278, 394], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    88: [
        {'parts': [{'rect': [47, 33, 138, 110], 'label': 'Provinsi dan alamat'}, {'rect': [138, 33, 278, 110], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 110, 138, 187], 'label': 'Provinsi dan alamat'}, {'rect': [138, 110, 278, 187], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 187, 138, 229], 'label': 'Provinsi dan alamat'}, {'rect': [138, 187, 278, 229], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 229, 138, 303], 'label': 'Provinsi dan alamat'}, {'rect': [138, 229, 278, 303], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 303, 138, 390], 'label': 'Provinsi dan alamat'}, {'rect': [138, 303, 278, 390], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    89: [
        {'parts': [{'rect': [47, 79, 138, 155], 'label': 'Provinsi dan alamat'}, {'rect': [138, 79, 278, 155], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 155, 138, 228], 'label': 'Provinsi dan alamat'}, {'rect': [138, 155, 278, 228], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 228, 138, 281], 'label': 'Provinsi dan alamat'}, {'rect': [138, 228, 278, 281], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 281, 138, 325], 'label': 'Provinsi dan alamat'}, {'rect': [138, 281, 278, 325], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 325, 138, 390], 'label': 'Provinsi dan alamat'}, {'rect': [138, 325, 278, 390], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    90: [
        {'parts': [{'rect': [47, 31, 138, 86], 'label': 'Provinsi dan alamat'}, {'rect': [138, 31, 278, 86], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 86, 138, 138], 'label': 'Provinsi dan alamat'}, {'rect': [138, 86, 278, 138], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 138, 138, 183], 'label': 'Provinsi dan alamat'}, {'rect': [138, 138, 278, 183], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 183, 138, 236], 'label': 'Provinsi dan alamat'}, {'rect': [138, 183, 278, 236], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 236, 138, 288], 'label': 'Provinsi dan alamat'}, {'rect': [138, 236, 278, 288], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 288, 138, 390], 'label': 'Provinsi dan alamat'}, {'rect': [138, 288, 278, 390], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    91: [
        {'parts': [{'rect': [47, 79, 138, 120], 'label': 'Provinsi dan alamat'}, {'rect': [138, 79, 278, 120], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 120, 138, 173], 'label': 'Provinsi dan alamat'}, {'rect': [138, 120, 278, 173], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
        {'parts': [{'rect': [47, 173, 138, 390], 'label': 'Provinsi dan alamat'}, {'rect': [138, 173, 278, 390], 'label': 'Informasi kontak dalam buku'}], 'phase': 'UMUM', 'subsection': 'Kontak historis dari buku'},
    ],
    92: [
        {'parts': [{'rect': [20, 60, 270, 76]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 76, 270, 92]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 92, 270, 124]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 124, 270, 140]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 140, 270, 156]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 156, 270, 172]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 172, 270, 188]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 188, 270, 204]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 204, 270, 220]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 220, 270, 236]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 236, 270, 252]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 252, 270, 284]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 284, 270, 300]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
        {'parts': [{'rect': [20, 300, 270, 317]}], 'phase': 'UMUM', 'subsection': 'Daftar singkatan'},
    ],
    53: [
        {'parts': [{'rect': [20, 87, 100, 169]}], 'phase': 'SAAT', 'subsection': 'Ilustrasi penghematan air', 'text_override': 'Kurangi pemakaian air yang berlebih bila tidak digunakan. MATIKAN AIR', 'note': 'Menyusun urutan slogan dua bagian yang terletak mengelilingi ilustrasi keran.'},
        {'parts': [{'rect': [20, 258, 275, 286]}], 'phase': 'SAAT'},
        {'parts': [{'rect': [20, 299, 275, 318]}], 'phase': 'SAAT'},
        {'parts': [{'rect': [20, 336, 275, 355]}], 'phase': 'SAAT'},
        {'parts': [{'rect': [20, 368, 275, 395]}], 'phase': 'SAAT'},
    ],
    59: [
        {'parts': [{'rect': [35, 128, 72, 169]}], 'phase': 'UMUM', 'subsection': 'Call center yang dicantumkan buku', 'quality_flags': ['HISTORICAL_RECORD']},
        {'parts': [{'rect': [18, 196, 275, 274]}], 'phase': 'UMUM', 'subsection': 'Latar historis pandemi dalam buku'},
        {'parts': [{'rect': [18, 289, 275, 346]}], 'phase': 'UMUM', 'subsection': 'Latar historis pandemi dalam buku'},
        {'parts': [{'rect': [18, 347, 275, 394]}], 'phase': 'UMUM', 'subsection': 'Latar historis pandemi dalam buku'},
    ],
    82: [
        {'parts': [{'rect': [15, 88, 78, 119]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [79, 88, 139, 119]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [140, 88, 206, 119]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [207, 88, 275, 119]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [15, 146, 78, 177]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [79, 146, 139, 177]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [140, 146, 206, 177]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [207, 146, 275, 177]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [20, 222, 80, 254]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [86, 222, 137, 254]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [141, 205, 198, 230]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [205, 205, 273, 238]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [20, 282, 91, 324]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [120, 296, 181, 320]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [199, 295, 274, 326]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
        {'parts': [{'rect': [147, 334, 211, 359]}], 'phase': 'UMUM', 'subsection': 'Label rambu dan papan informasi dalam buku'},
    ],
    83: [
        {'parts': [{'rect': [50, 158, 90, 174]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [106, 158, 172, 174]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [195, 158, 240, 174]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [45, 259, 102, 276]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [129, 255, 172, 279]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [200, 259, 246, 276]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [45, 359, 103, 376]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [130, 359, 163, 376]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
        {'parts': [{'rect': [195, 359, 250, 376]}], 'phase': 'UMUM', 'subsection': 'Daftar aplikasi gawai yang dicantumkan buku'},
    ],
}

## 4. Membersihkan teks dan membaca area khusus

Pembersihan hanya menormalkan karakter dan spasi. Angka, satuan, negasi, kondisi, dan isi kalimat dipertahankan. Teks pada area khusus disusun berdasarkan baris, lalu posisi kata dari kiri ke kanan.

In [4]:
def clean(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\u00ad", "")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def phase_at(page_number, y):
    phase = DEFAULT_PHASE.get(page_number, "UMUM")
    for start, label in PHASE_CHANGES.get(page_number, []):
        if y >= start:
            phase = label
    return phase


def word_left(word):
    return word[0]


def region_text(words, rect):
    lines = {}
    for word in words:
        x0, y0, x1, y1, text, block, line, word_number = word
        center_x = (x0 + x1) / 2
        center_y = (y0 + y1) / 2
        if rect[0] <= center_x < rect[2] and rect[1] <= center_y < rect[3]:
            line_id = (block, line)
            if line_id not in lines:
                lines[line_id] = []
            lines[line_id].append(word)

    # Simpan posisi setiap baris agar urutan baca tidak mengikuti urutan objek PDF.
    ordered_lines = []
    for index, line_words in enumerate(lines.values()):
        top = line_words[0][1]
        left = line_words[0][0]
        for word in line_words:
            top = min(top, word[1])
            left = min(left, word[0])
        ordered_lines.append((round(top, 1), left, index, line_words))
    ordered_lines.sort()

    text_lines = []
    for top, left, index, line_words in ordered_lines:
        line_words.sort(key=word_left)
        text_words = []
        for word in line_words:
            text_words.append(word[4])
        text_lines.append(" ".join(text_words))
    return clean(" ".join(text_lines))


def special_units(page, page_number):
    words = page.get_text("words")
    result = []
    for spec in LAYOUT[page_number]:
        pieces = []
        has_label = False
        for part in spec["parts"]:
            text = region_text(words, part["rect"])
            if not text:
                raise ValueError(f"Region kosong di PDF halaman {page_number}: {part['rect']}")
            if part.get("label"):
                text = part["label"] + ": " + text
                has_label = True
            pieces.append(text)

        if has_label:
            text = "; ".join(pieces)
        else:
            text = " ".join(pieces)
        if "text_override" in spec:
            text = spec["text_override"]

        row = spec.copy()
        row["text"] = clean(text)
        result.append(row)
    return result

## 5. Membaca halaman biasa

Halaman tanpa aturan khusus dibaca per blok dan baris. Judul, nomor daftar yang berdiri sendiri, serta teks footer dilewati. Baris dipisah menjadi unit baru saat fase berubah atau daftar baru dimulai.

In [5]:
SKIP_TEXT = {
    "Prabencana", "Saat Bencana", "Pascabencana", "SIAGA BENCANA",
    "KESIAPSIAGAAN", "TANGGAP TANGKAS TANGGUH", "MENGHADAPI BENCANA",
}


def line_position(line):
    return (line["bbox"][1], line["bbox"][0])


def unit_position(unit):
    return (unit["y"], unit["x"])


def normal_units(page, page_number):
    result = []
    # Abaikan gambar karena yang dibaca hanya teks.
    flags = pymupdf.TEXTFLAGS_DICT & ~pymupdf.TEXT_PRESERVE_IMAGES
    page_data = page.get_text("dict", flags=flags)
    for block in page_data["blocks"]:
        if block["type"] != 0:
            continue

        runs = []
        lines = sorted(block["lines"], key=line_position)
        for line in lines:
            span_texts = []
            size = 0
            for span in line["spans"]:
                span_texts.append(span["text"])
                size = max(size, span["size"])
            text = clean("".join(span_texts))
            y = line["bbox"][1]

            if not text or y >= 415:
                continue
            if text in SKIP_TEXT or text in SKIP_EXACT.get(page_number, []):
                continue
            if re.fullmatch(r"\d+[.)]?", text):
                continue
            if text in HEADINGS and size > 10:
                continue

            phase = phase_at(page_number, y)
            start_new_unit = False
            if not runs:
                start_new_unit = True
            elif phase != runs[-1]["phase"]:
                start_new_unit = True
            elif re.match(r"(?:\d{1,2}[.)]|[•●])\s", text):
                start_new_unit = True

            if start_new_unit:
                runs.append({
                    "parts": [],
                    "phase": phase,
                    "y": y,
                    "x": line["bbox"][0],
                })
            runs[-1]["parts"].append(text)

        for row in runs:
            row["text"] = clean(" ".join(row["parts"]))
            result.append(row)

    result.sort(key=unit_position)
    return result

## 6. Menambahkan metadata dan mengumpulkan unit

Informasi historis atau bagian sumber yang ambigu diberi flag untuk diperiksa. Label konteks disimpan terpisah dari teks buku. ID unit memakai halaman, urutan, dan hash teks agar tetap sama saat notebook dijalankan ulang dengan input yang sama.

In [6]:
def quality_flags(page_number, text, spec):
    flags = set(spec.get("quality_flags", []))
    if 59 <= page_number <= 62:
        flags.update(["HISTORICAL_HEALTH_GUIDANCE", "TIME_SENSITIVE", "SOURCE_ATTRIBUTION_REQUIRED"])
    if 83 <= page_number <= 91:
        flags.update(["HISTORICAL_RECORD", "TIME_SENSITIVE", "SOURCE_ATTRIBUTION_REQUIRED"])
    if 71 <= page_number <= 75:
        flags.update(["SOURCE_ATTRIBUTION_REQUIRED", "NO_ABSOLUTE_PROTECTION_CLAIM"])
    if page_number == 28 and (">10.000" in text or "ratusan kilometer" in text):
        flags.add("DUBIOUS_SOURCE")
    if page_number == 45 and "liquefaction" in text:
        flags.add("DUBIOUS_SOURCE")
    if page_number == 57 and "50 kaki" in text:
        flags.update(["SOURCE_SENSITIVE", "SOURCE_ATTRIBUTION_REQUIRED"])
    if page_number == 62 and "perkantoran atau" in text:
        flags.add("INCOMPLETE_SOURCE")
    return sorted(flags)


def retrieval_text(row):
    label = f"{row['section']} | {row['subsection']}"
    if row["context"]:
        label += f" | {row['context']}"
    return label + "\n" + row["text"]

In [7]:
units = []
with pymupdf.open(stream=pdf_bytes, filetype="pdf") as pdf:
    if len(pdf) != 98:
        raise ValueError("Salinan sumber harus mempunyai 98 halaman PDF.")

    for number, page in enumerate(pdf, 1):
        section = None
        for item in SECTIONS:
            start_page, end_page, title, topic, material = item
            if start_page <= number <= end_page:
                section = item
                break
        if section is None:
            continue

        start_page, end_page, title, topic, material = section
        if number in LAYOUT:
            specs = special_units(page, number)
        else:
            specs = normal_units(page, number)

        for order, spec in enumerate(specs, 1):
            text = spec["text"]
            phase = spec["phase"]
            context = PLANT_CONTEXT.get(number, "")
            if "context" in spec:
                context = spec["context"]
            if number == 18:
                context = "Di dalam bangunan seperti rumah sekolah ataupun bangunan bertingkat"
            if number == 19 and spec.get("y", 999) < 185:
                context = "Di dalam mobil"

            text_hash = hashlib.sha256(text.encode()).hexdigest()[:8]
            unit_id = f"BNPB-{pdf_sha256[:8]}-p{number:03d}-u{order:03d}-{text_hash}"
            row = {
                "id": unit_id,
                "text": text,
                "topic": topic,
                "phase": phase,
                "section": title,
                "subsection": spec.get("subsection", phase),
                "context": context,
                "material": material,
                "source": {
                    "file": PDF_PATH.name,
                    "pdf_page": number,
                    "book_page": number - 2,
                },
                "flags": quality_flags(number, text, spec),
                "review_status": "pending_group_review",
            }
            if spec.get("note"):
                row["curation_note"] = spec["note"]
            row["retrieval_text"] = retrieval_text(row)
            units.append(row)

known_ids = set()
for unit in units:
    known_ids.add(unit["id"])
for unit_id in EXCLUDE_IDS:
    if unit_id not in known_ids:
        raise ValueError(f"EXCLUDE_IDS berisi ID yang tidak ditemukan: {unit_id}")

active_units = []
for unit in units:
    if unit["id"] not in EXCLUDE_IDS:
        active_units.append(unit)
units = active_units

topics = set()
pages = set()
for unit in units:
    topics.add(unit["topic"])
    pages.add(unit["source"]["pdf_page"])
for topic in HAZARDS:
    if topic not in topics:
        raise ValueError(f"Topik {topic} tidak ada dalam unit aktif. Periksa EXCLUDE_IDS.")

print(f"Unit bukti aktif: {len(units)}")
print(f"Halaman pengetahuan: {len(pages)}")

Unit bukti aktif: 455
Halaman pengetahuan: 79


## 7. Membuat chunk RAG

Chunk menggabungkan unit utuh dengan halaman, topik, fase, subbagian, konteks, jenis materi, dan flag yang sama. Tokenizer E5 menghitung `"passage: " + retrieval_text`, termasuk special tokens. Batas setiap chunk adalah `MAX_TOKENS`.

Jika satu unit sudah melebihi batas, proses berhenti agar unit atau batasnya dapat diperiksa. Notebook hanya memuat tokenizer, tanpa model embedding atau LLM.

In [8]:
tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    revision=TOKENIZER_REVISION,
    trust_remote_code=False,
    use_fast=True,
)
if MAX_TOKENS < 1 or MAX_TOKENS > tokenizer.model_max_length:
    raise ValueError(f"MAX_TOKENS harus 1 sampai {tokenizer.model_max_length}.")


def token_count(text):
    token_ids = tokenizer.encode(
        "passage: " + text,
        add_special_tokens=True,
        truncation=False,
    )
    return len(token_ids)


def group_key(row):
    return (
        row["source"]["pdf_page"],
        row["topic"],
        row["phase"],
        row["subsection"],
        row["context"],
        row["material"],
        tuple(row["flags"]),
    )


def make_chunk(items):
    first = items[0]
    row = {}
    metadata_fields = [
        "topic", "phase", "section", "subsection", "context",
        "material", "source", "flags", "review_status",
    ]
    for field in metadata_fields:
        row[field] = first[field]

    unit_ids = []
    texts = []
    for unit in items:
        unit_ids.append(unit["id"])
        texts.append(unit["text"])

    row["unit_ids"] = unit_ids
    row["text"] = "\n".join(texts)
    row["retrieval_text"] = retrieval_text(row)
    row["token_count"] = token_count(row["retrieval_text"])

    ids_json = json.dumps(unit_ids)
    chunk_hash = hashlib.sha256(ids_json.encode()).hexdigest()[:16]
    row["id"] = "chunk-" + chunk_hash
    return row


chunks = []
current_units = []
for unit in units:
    single_chunk = make_chunk([unit])
    if single_chunk["token_count"] > MAX_TOKENS:
        raise ValueError(f"Unit {unit['id']} melebihi {MAX_TOKENS} token. Periksa unit atau naikkan batas token.")

    if current_units:
        if group_key(unit) != group_key(current_units[0]):
            chunks.append(make_chunk(current_units))
            current_units = []
        else:
            candidate_units = current_units + [unit]
            candidate_chunk = make_chunk(candidate_units)
            if candidate_chunk["token_count"] > MAX_TOKENS:
                chunks.append(make_chunk(current_units))
                current_units = []

    current_units.append(unit)

if current_units:
    chunks.append(make_chunk(current_units))

largest_chunk = 0
for chunk in chunks:
    largest_chunk = max(largest_chunk, chunk["token_count"])
print(f"Chunk RAG: {len(chunks)}")
print(f"Token terbesar: {largest_chunk}")

Chunk RAG: 128
Token terbesar: 348


## 8. Memeriksa hasil dan memberi versi korpus

Pemeriksaan memastikan ID unik, nomor halaman sesuai, setiap unit muncul tepat sekali dalam chunk, dan batas token terpenuhi. Versi korpus disimpan sebagai metadata pada setiap baris JSONL dan di `corpus.md`.

In [9]:
by_id = {}
for unit in units:
    unit_id = unit["id"]
    if unit_id in by_id:
        raise ValueError(f"ID unit duplikat: {unit_id}")
    by_id[unit_id] = unit

    if not unit["text"]:
        raise ValueError(f"Teks unit kosong: {unit_id}")
    if unit["phase"] not in {"UMUM", "PRA", "SAAT", "PASCA"}:
        raise ValueError(f"Fase tidak valid: {unit_id}")
    if unit["source"]["book_page"] != unit["source"]["pdf_page"] - 2:
        raise ValueError(f"Nomor halaman buku tidak cocok: {unit_id}")
    if unit["topic"] in {"KARHUTLA", "COVID"} and unit["phase"] == "PASCA":
        raise ValueError("Fase pascabencana tidak tersedia sebagai subbagian topik ini.")

chunk_ids = set()
represented = Counter()
for chunk in chunks:
    if chunk["id"] in chunk_ids:
        raise ValueError(f"ID chunk duplikat: {chunk['id']}")
    chunk_ids.add(chunk["id"])

    texts = []
    for unit_id in chunk["unit_ids"]:
        unit = by_id[unit_id]
        if group_key(unit) != group_key(chunk):
            raise ValueError("Chunk mencampur halaman, fase, atau konteks.")
        texts.append(unit["text"])
        represented[unit_id] += 1

    if chunk["text"] != "\n".join(texts):
        raise ValueError("Isi chunk tidak cocok dengan unit sumber.")
    if chunk["token_count"] != token_count(chunk["retrieval_text"]):
        raise ValueError("Hitungan token chunk tidak cocok.")
    if chunk["token_count"] > MAX_TOKENS:
        raise ValueError("Chunk melampaui batas token.")

for unit_id in by_id:
    if represented[unit_id] != 1:
        raise ValueError(f"Unit harus muncul tepat sekali dalam chunk: {unit_id}")

# Hash mengidentifikasi hasil yang sama tanpa memerlukan file manifest.
tokenizer_json = tokenizer.backend_tokenizer.to_str()
tokenizer_hash = hashlib.sha256(tokenizer_json.encode()).hexdigest()
version_data = {
    "pdf_sha256": pdf_sha256,
    "tokenizer": tokenizer_hash,
    "revision": TOKENIZER_REVISION,
    "max_tokens": MAX_TOKENS,
    "units": units,
    "chunks": chunks,
}
version_json = json.dumps(version_data, ensure_ascii=False, sort_keys=True)
version_hash = hashlib.sha256(version_json.encode()).hexdigest()[:16]
corpus_version = "bnpb-" + version_hash
for unit in units:
    unit["corpus_version"] = corpus_version
for chunk in chunks:
    chunk["corpus_version"] = corpus_version

print("Validasi selesai.")
print(f"Versi: {corpus_version}")

Validasi selesai.
Versi: bnpb-ee9981b4e47a331d


## 9. Menyimpan tiga file hasil

Setelah validasi selesai, simpan passage klasik, chunk RAG, dan Markdown untuk review. Menjalankan ulang sel ini akan menimpa ketiga file tersebut.

In [10]:
markdown_lines = [
    "# Korpus QA Bencana BNPB",
    "",
    f"Versi: `{corpus_version}`",
    "",
    "Sumber: Buku Saku Tanggap Tangkas Tangguh Menghadapi Bencana, BNPB.",
    f"SHA256 PDF: `{pdf_sha256}`",
    "Status: pending_group_review. Label konteks adalah anotasi, bukan kutipan tambahan.",
    "Informasi historis/ambigu mengikuti flag sumber dan belum menjadi informasi terkini.",
    "",
]
previous_page = None
for unit in units:
    source = unit["source"]
    if source["pdf_page"] != previous_page:
        page_heading = f"## {unit['section']} — Buku {source['book_page']} · PDF {source['pdf_page']}"
        markdown_lines.append(page_heading)
        markdown_lines.append("")
        previous_page = source["pdf_page"]

    heading = "### " + unit["subsection"]
    if unit["context"]:
        heading += " · " + unit["context"]

    if unit["flags"]:
        flag_text = ", ".join(unit["flags"])
    else:
        flag_text = "tidak ada"

    markdown_lines.append(heading)
    markdown_lines.append(f"ID: `{unit['id']}` · Fase: {unit['phase']} · Flag: {flag_text}")
    markdown_lines.append("")
    markdown_lines.append(unit["text"])
    markdown_lines.append("")
    if unit.get("curation_note"):
        markdown_lines.append(f"Catatan transkripsi: {unit['curation_note']}")
        markdown_lines.append("")

if OUTPUT_DIR.is_symlink():
    raise ValueError("OUTPUT_DIR harus berupa folder biasa.")
if OUTPUT_DIR.exists() and not OUTPUT_DIR.is_dir():
    raise ValueError("OUTPUT_DIR harus berupa folder.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

classic_path = OUTPUT_DIR / "classic_passages.jsonl"
with classic_path.open("w", encoding="utf-8") as output_file:
    for unit in units:
        line = json.dumps(unit, ensure_ascii=False)
        output_file.write(line + "\n")

rag_path = OUTPUT_DIR / "rag_chunks.jsonl"
with rag_path.open("w", encoding="utf-8") as output_file:
    for chunk in chunks:
        line = json.dumps(chunk, ensure_ascii=False)
        output_file.write(line + "\n")

markdown_path = OUTPUT_DIR / "corpus.md"
markdown_text = "\n".join(markdown_lines)
markdown_path.write_text(markdown_text, encoding="utf-8")

print("Selesai. Tiga file yang dihasilkan:")
print(classic_path)
print(rag_path)
print(markdown_path)
print(f"{len(units)} unit | {len(chunks)} chunk | semua chunk <= {MAX_TOKENS} token")

Selesai. Tiga file yang dihasilkan:
/home/fast9/Secondary/CS5/nlp/ta-nlp/corpus/output/classic_passages.jsonl
/home/fast9/Secondary/CS5/nlp/ta-nlp/corpus/output/rag_chunks.jsonl
/home/fast9/Secondary/CS5/nlp/ta-nlp/corpus/output/corpus.md
455 unit | 128 chunk | semua chunk <= 350 token


## Memakai hasil

- **QA tanpa LLM:** muat `classic_passages.jsonl`. Gunakan `retrieval_text` untuk pencarian dan `text` untuk ekstraksi jawaban.
- **QA RAG:** muat `rag_chunks.jsonl`. Input embedding E5 adalah `"passage: " + row["retrieval_text"]`, dengan satu prefix. `unit_ids` menunjuk ke `id` pada passage klasik.
- **Review kelompok:** baca `corpus.md` dan cocokkan dengan halaman sumber. Periksa flag `DUBIOUS_SOURCE`, `SOURCE_AMBIGUOUS`, dan `INCOMPLETE_SOURCE` sebelum evaluasi final. Gunakan `EXCLUDE_IDS` untuk menyaring unit pada ketiga output saat **Run all**.

Korpus ini berisi pengetahuan, bukan pertanyaan, gold answer, atau indeks embedding. Simpan kunci QA evaluasi secara terpisah. Di Colab, unduh tiga file dari folder `output/` melalui panel **Files**.

Referensi teknis: [PyMuPDF](https://pymupdf.readthedocs.io/en/latest/recipes-text.html), [model card multilingual E5](https://huggingface.co/intfloat/multilingual-e5-small), [buku BNPB](https://bnpb.go.id/storage/app/media/Buku%20BNPB/Buku%20Saku%20Bencana%20BNPB.pdf).